# Generate Splits for BioBERT 

In [10]:
import sys, os, json, random
from pathlib import Path 

PROJECT_ROOT = Path("/Users/robertagarcia/Desktop/learning/bert_symptom_ner")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))


In [11]:

# Correct - ensure data/biobert_splits under PROJECT_ROOT/v03/data/
SPLIT_DIR = PROJECT_ROOT / "v03" / "data" / "biobert_splits"
os.makedirs(SPLIT_DIR, exist_ok=True)
INPUT_FILE = PROJECT_ROOT / "v03" / "data" / "data_wordpiece_tokenized_biobert.jsonl"
TRAIN_FILE = SPLIT_DIR / "train.jsonl"
VAL_FILE = SPLIT_DIR / "val.jsonl"
TEST_FILE = SPLIT_DIR / "test.jsonl"

with open(INPUT_FILE, "r") as f:
    data = [json.loads(line) for line in f]

# Shuffle
random.shuffle(data)

# biobert_splits
n = len(data)
train_end = int(0.8*n)  # 80% train data 
val_end = int(0.9*n)    # 10% val data
train_data = data[:train_end]
val_data   = data[train_end:val_end]
test_data  = data[val_end:] 

# Save helper
def save_jsonl(path, dataset):
    with open(path, "w") as f:
        for row in dataset:
            f.write(json.dumps(row) + "\n")

# Save splits
save_jsonl(TRAIN_FILE, train_data)
save_jsonl(VAL_FILE, val_data)
save_jsonl(TEST_FILE, test_data)

print(f"Train: {len(train_data)}")
print(f"Val:   {len(val_data)}")
print(f"Test:  {len(test_data)}")


Train: 85013
Val:   10627
Test:  10627


# Upload Datasets to Huggingface

In [12]:
from datasets import load_dataset

data_files = {
"train": f"{TRAIN_FILE}",
"validation": f"{VAL_FILE}",
"test" : f"{TEST_FILE}"
}

dataset = load_dataset("json", data_files=data_files)
print(dataset)

Generating train split: 85013 examples [00:00, 639456.19 examples/s]
Generating validation split: 10627 examples [00:00, 1248924.56 examples/s]
Generating test split: 10627 examples [00:00, 832422.00 examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'word_tokens', 'word_labels', 'tokens', 'input_ids', 'token_labels', 'token_label_ids'],
        num_rows: 85013
    })
    validation: Dataset({
        features: ['text', 'word_tokens', 'word_labels', 'tokens', 'input_ids', 'token_labels', 'token_label_ids'],
        num_rows: 10627
    })
    test: Dataset({
        features: ['text', 'word_tokens', 'word_labels', 'tokens', 'input_ids', 'token_labels', 'token_label_ids'],
        num_rows: 10627
    })
})


In [13]:
dataset.push_to_hub("Rogarcia18/symptoms_ner_v03_biobert", private=False)

Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00,  5.72ba/s]
Processing Files (1 / 1): 100%|██████████| 6.19MB / 6.19MB, 3.09MB/s  
New Data Upload: 100%|██████████| 6.19MB / 6.19MB, 3.09MB/s  
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 44.32ba/s]
Processing Files (1 / 1): 100%|██████████|  812kB /  812kB,  661kB/s  
New Data Upload: 100%|██████████|  812kB /  812kB,  661kB/s  
Creating parquet from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 37.54ba/s]
Processing Files (1 / 1): 100%|██████████|  819kB /  819kB,  719kB/s  
New Data Upload: 100%|██████████|  819kB /  819kB,  719kB/s  
Uploading the dataset shards: 100%|██████████| 1/1 [00:01<00:00,  1.24s/ shards]


CommitInfo(commit_url='https://huggingface.co/datasets/Rogarcia18/symptoms_ner_v03_biobert/commit/ca3d19fa422518e9dc004698bc22c5a53a38a9a2', commit_message='Upload dataset', commit_description='', oid='ca3d19fa422518e9dc004698bc22c5a53a38a9a2', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/Rogarcia18/symptoms_ner_v03_biobert', endpoint='https://huggingface.co', repo_type='dataset', repo_id='Rogarcia18/symptoms_ner_v03_biobert'), pr_revision=None, pr_num=None)

In [14]:

from huggingface_hub import HfApi, upload_file, hf_hub_download

# Load the label mappings
with open(f"{PROJECT_ROOT}/v03/data/id2label.json", "r") as f:
    id2label = json.load(f)
with open(f"{PROJECT_ROOT}/v03/data/label2id.json", "r") as f:
    label2id = json.load(f)

# Initialize the Hugging Face API
api = HfApi()
repo_id = "Rogarcia18/symptoms_ner_v03_biobert"
# Check existing files in the repository (for information only)
existing_files = api.list_repo_files(repo_id=repo_id, repo_type="dataset")
print(f"Existing files in repository: {existing_files}")

Existing files in repository: ['.gitattributes', 'README.md', 'data/test-00000-of-00001.parquet', 'data/train-00000-of-00001.parquet', 'data/validation-00000-of-00001.parquet', 'id2label.json', 'label2id.json']


In [6]:
id2label,label2id

({'0': 'B-SYMPTOM_NEG',
  '1': 'B-SYMPTOM_POS',
  '2': 'I-SYMPTOM_NEG',
  '3': 'I-SYMPTOM_POS',
  '4': 'O'},
 {'B-SYMPTOM_NEG': 0,
  'B-SYMPTOM_POS': 1,
  'I-SYMPTOM_NEG': 2,
  'I-SYMPTOM_POS': 3,
  'O': 4})

In [15]:
# Upload id2label.json
print("Uploading id2label.json...")
upload_file(
    path_or_fileobj=f"{PROJECT_ROOT}/v03/data/id2label.json",
    path_in_repo="id2label.json",
    repo_id=repo_id,
    repo_type="dataset",
)
print("✓ id2label.json uploaded successfully")

# Upload label2id.json
print("Uploading label2id.json...")
upload_file(
    path_or_fileobj=f"{PROJECT_ROOT}/v03/data/label2id.json",
    path_in_repo="label2id.json",
    repo_id=repo_id,
    repo_type="dataset",
)
print("✓ label2id.json uploaded successfully")

Uploading id2label.json...


No files have been modified since last commit. Skipping to prevent empty commit.
No files have been modified since last commit. Skipping to prevent empty commit.


✓ id2label.json uploaded successfully
Uploading label2id.json...
✓ label2id.json uploaded successfully


**Test downloading the data (dataset is too large)**

In [8]:
# from datasets import load_dataset

# dataset = load_dataset(repo_id, repo_type="dataset")
# print(dataset)

In [16]:
# Download and load id2label.json from the hub
id2label_path = hf_hub_download(
    repo_id=repo_id,
    filename="id2label.json",
    repo_type="dataset"
)
with open(id2label_path, "r") as f:
    id2label = json.load(f)
print(f"✓ Loaded id2label.json from hub")
print(f"  Total labels: {len(id2label)}")
print(f"  First 5 labels: {dict(list(id2label.items())[:5])}")

# Download and load label2id.json from the hub
label2id_path = hf_hub_download(
    repo_id=repo_id,
    filename="label2id.json",
    repo_type="dataset"
)
with open(label2id_path, "r") as f:
    label2id = json.load(f)
print(f"\n✓ Loaded label2id.json from hub")
print(f"  Total labels: {len(label2id)}")
print(f"  First 5 labels: {dict(list(label2id.items())[:5])}")

# Now you can use id2label and label2id in your code!

✓ Loaded id2label.json from hub
  Total labels: 5
  First 5 labels: {'0': 'B-SYMPTOM_NEG', '1': 'B-SYMPTOM_POS', '2': 'I-SYMPTOM_NEG', '3': 'I-SYMPTOM_POS', '4': 'O'}

✓ Loaded label2id.json from hub
  Total labels: 5
  First 5 labels: {'B-SYMPTOM_NEG': 0, 'B-SYMPTOM_POS': 1, 'I-SYMPTOM_NEG': 2, 'I-SYMPTOM_POS': 3, 'O': 4}
